# Random Forest Taxi Destination Notebook

Cached-data Kaggle workflow for Porto taxi destination prediction: load prebuilt preprocessing artifacts generated by `code/build_preprocessed_datasets.py`, train a Random Forest model, validate with Haversine distance, and export a submission.


## Cached Data Paths

These model notebooks do not rebuild raw preprocessing. They expect the cached preprocessed Kaggle Dataset generated by `code/build_preprocessed_datasets.py`, including `metadata.json`, `validation/`, and `final/` matrices. Attach that dataset on Kaggle or set `PREPROCESSED_DATA_DIR`; model artifacts and submissions are written under `/kaggle/working`.


In [ ]:
import os
from pathlib import Path

PREPROCESSED_DATA_DIR = Path(
    os.environ.get(
        "PREPROCESSED_DATA_DIR",
        "/kaggle/input/datasets/louishogge/preprocessed-taxi-dataset/data",
    )
)
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
ENSEMBLE_EXPORT_DIR = OUTPUT_DIR / "ensemble_inputs"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessed data directory:", PREPROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Submission directory:", SUBMISSION_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())
print("Ensemble export directory:", ENSEMBLE_EXPORT_DIR.resolve())


## Imports

Load dependencies used by cached data loading, modeling, tuning, and plots.


In [ ]:
import gc
import json
import random
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid


## Hyperparameters

Set the reproducible run configuration and active model parameters.

| Mode | Required settings | Behavior |
|---|---|---|
| One-candidate tuning | RUN_HYPERPARAMETER_TUNING=True, set TUNING_CANDIDATE_INDEX, other expensive stages False | Evaluate exactly one deterministic finite candidate for a parallel Kaggle run. |
| Validation refit | RUN_VALIDATION_REFIT=True, other expensive stages False | Refit the saved model on the validation-training split, export aligned ensemble predictions, and render full diagnostics. |
| Final fit, lightweight bundle | RUN_FINAL_FIT=True, serialization False | Fit the saved model on full training data, write a reconstruction bundle, and create the submission. |
| Final fit with estimator | RUN_FINAL_FIT=True, serialization True | Secure the lightweight bundle and submission first, then serialize the fitted forest. |

Only one expensive stage may be enabled per Kaggle run. With all expensive stages disabled, the notebook runs in inspection mode and writes no outputs.

In [ ]:
SEED = 42
MODEL_NAME = "Random Forest"
MODEL_SLUG = "random_forest"

RUN_HYPERPARAMETER_TUNING = False
RUN_VALIDATION_REFIT = False
RUN_FINAL_FIT = True
SERIALIZE_RANDOM_FOREST_MODEL = False

# Random Forest tuning is one deterministic candidate per parallel Kaggle run.
TUNING_CANDIDATE_INDEX = None
TUNING_MAX_TRIALS = 1

enabled_expensive_stages = sum([
    RUN_HYPERPARAMETER_TUNING,
    RUN_VALIDATION_REFIT,
    RUN_FINAL_FIT,
])
if enabled_expensive_stages > 1:
    raise ValueError(
        "Enable only one of RUN_HYPERPARAMETER_TUNING, RUN_VALIDATION_REFIT, "
        "or RUN_FINAL_FIT for a Kaggle run."
    )
if SERIALIZE_RANDOM_FOREST_MODEL and not RUN_FINAL_FIT:
    raise ValueError(
        "SERIALIZE_RANDOM_FOREST_MODEL=True requires RUN_FINAL_FIT=True."
    )

WANDB_PROJECT = "machine-learning"
WANDB_ENTITY = "ml-l"
WANDB_SECRET_NAME = "wandb-api-key"

SUBMISSION_FILE = f"submission_{MODEL_SLUG}.csv"
BEST_PARAMS_PATH = ARTIFACT_DIR / f"best_params_{MODEL_SLUG}.json"
MODEL_BUNDLE_PATH = ARTIFACT_DIR / f"model_bundle_{MODEL_SLUG}.joblib"
VALIDATION_PREDICTIONS_PATH = ENSEMBLE_EXPORT_DIR / f"validation_predictions_{MODEL_SLUG}.parquet"
VALIDATION_PREDICTIONS_MANIFEST_PATH = ENSEMBLE_EXPORT_DIR / f"validation_predictions_{MODEL_SLUG}.manifest.json"
VALIDATION_EXPORT_DRIFT_TOLERANCE_KM = 0.001

BEST_MODEL_PARAMS = {
    "n_estimators": 700,
    "max_depth": 100,
    "min_samples_split": 5,
    "min_samples_leaf": 5,
    "max_features": 0.5,
    "bootstrap": True,
}

BEST_MODEL_RUN_SUMMARY = {
    "reviewed_on": "2026-07-10",
    "source": "W&B full-history export plus completed Kaggle validation-refit, final-fit, and local submission results",
    "selection_status": "selected_safe_final_model",
    "trial_index": 1,
    "selected_run_trial_count": 1,
    "equivalent_best_trial_indices": [1],
    "run_name": "random_forest-rf_fixed_07_n700_depth100_mf05-fixed_grid-trial-001",
    "run_variant": "rf_fixed_07_n700_depth100_mf05",
    "tuner_type": "fixed_grid",
    "tested_configurations": 12,
    "expected_configurations": 12,
    "all_configurations_tested": True,
    "historical_search_execution_mode": "parallel_single_variant_runs",
    "configuration_coverage_scope": "combined_wandb_history",
    "configuration_coverage_by_tuner": {
        "fixed_grid": 9,
        "optuna_tpe": 3,
    },
    "validation_haversine_mean_km": 2.1194417721912764,
    "validation_haversine_median_km": 1.2038758188442602,
    "validation_haversine_std_km": 3.3611060963470516,
    "validation_haversine_p90_km": 4.615561284377124,
    "train_haversine_mean_km": 1.3420295612620714,
    "train_haversine_median_km": 0.751898086335455,
    "train_haversine_std_km": 2.163060594902953,
    "train_haversine_p90_km": 2.9722432338090625,
    "generalization_gap_km": 0.777412210929205,
    "fit_seconds": 15244.51231622696,
    "diagnostic_refit_fit_seconds": 25546.143852710724,
    "absolute_best_run_name": "random_forest-rf_fixed_09_n700_depthnone_mf06-fixed_grid-trial-001",
    "absolute_best_model_params": {
        "n_estimators": 700,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 5,
        "max_features": 0.6,
        "bootstrap": True,
    },
    "absolute_best_validation_haversine_mean_km": 2.119220536434474,
    "absolute_best_validation_haversine_p90_km": 4.61800822349927,
    "absolute_best_fit_seconds": 30336.201473236084,
    "validation_mean_delta_vs_absolute_best_km": 0.0002212357568027734,
    "validation_p90_improvement_vs_absolute_best_km": 0.002446939122146219,
    "fit_speedup_vs_absolute_best": 1.9899752018268788,
    "selection_rationale": "Selected for final fitting because it was only 0.221 m worse in validation mean than the absolute winner, had a 2.447 m better validation p90, and fit about 1.99x faster in the search runs.",
    "validation_refit_performed": True,
    "final_fit_performed": True,
    "local_submission_rank_by_public_mean": 2,
    "local_submission_all_mean_km": 2.3601805233836446,
    "local_submission_all_median_km": 1.2361316231613428,
    "local_submission_all_std_km": 3.1984189934568663,
    "local_submission_all_p90_km": 6.242961702816691,
    "local_submission_public_mean_km": 2.2061215959080807,
    "local_submission_private_mean_km": 2.5142394508592094,
    "local_submission_n_trips": 320,
}

# Historical 12-configuration region retained for reproducibility and targeted reruns.
TUNING_SEARCH_SPACE = {
    "n_estimators": [300, 500, 700],
    "max_depth": [100, None],
    "min_samples_split": [5],
    "min_samples_leaf": [5],
    "max_features": [0.5, 0.6],
    "bootstrap": [True],
}
TUNING_UNIQUE_CANDIDATE_COUNT = len(list(ParameterGrid(TUNING_SEARCH_SPACE)))

if RUN_HYPERPARAMETER_TUNING:
    if (
        not isinstance(TUNING_CANDIDATE_INDEX, int)
        or isinstance(TUNING_CANDIDATE_INDEX, bool)
        or not 1 <= TUNING_CANDIDATE_INDEX <= TUNING_UNIQUE_CANDIDATE_COUNT
    ):
        raise ValueError(
            "Set TUNING_CANDIDATE_INDEX to a one-based index between 1 and "
            f"{TUNING_UNIQUE_CANDIDATE_COUNT}."
        )

MODEL_PARAMS = BEST_MODEL_PARAMS.copy()

## Reproducibility

Seed random generators used by model training and tuning.


In [ ]:
def seed_everything(seed=SEED):
    """Seed Python and NumPy random generators."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
print("Seed:", SEED)


## Utility Constants

Define the cached target-column order used by the preprocessing helpers.

In [ ]:
TARGET_COLUMNS = ["END_LONG", "END_LAT"]

## Cached Preprocessed Dataset Helpers

Load and validate the shared preprocessed validation and final datasets.


In [ ]:
PREPROCESSED_SCHEMA_VERSION = 1
EXPECTED_CATEGORICAL_ENCODING_POLICY = {
    "CALL_TYPE": "one_hot",
    "ORIGIN_CALL": "missing_indicator_plus_log_count",
    "ORIGIN_STAND": "one_hot_missing_as_0",
    "TAXI_ID": "one_hot",
}


def require_existing_path(path, description):
    """Return an existing path or fail with an actionable Kaggle Dataset message."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}. "
            "Attach the preprocessed-taxi-dataset Kaggle Dataset or set PREPROCESSED_DATA_DIR."
        )
    return path


def load_preprocessed_metadata(base_dir=PREPROCESSED_DATA_DIR):
    """Load and validate cached preprocessing metadata."""
    metadata_path = require_existing_path(base_dir / "metadata.json", "preprocessed metadata")
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    schema_version = metadata.get("schema_version")
    if schema_version != PREPROCESSED_SCHEMA_VERSION:
        raise ValueError(
            f"Unsupported preprocessed dataset schema_version={schema_version!r}; "
            f"expected {PREPROCESSED_SCHEMA_VERSION}."
        )

    categorical_encoding_policy = metadata.get("constants", {}).get("categorical_encoding_policy")
    if categorical_encoding_policy != EXPECTED_CATEGORICAL_ENCODING_POLICY:
        raise ValueError(
            "Attached preprocessed dataset was built with an unexpected categorical encoding policy. "
            f"Expected {EXPECTED_CATEGORICAL_ENCODING_POLICY!r}, got {categorical_encoding_policy!r}. "
            "Regenerate the cached dataset with the current build_preprocessed_datasets.py script."
        )
    return metadata


def load_cached_frame(relative_path, base_dir=PREPROCESSED_DATA_DIR):
    """Load one cached Parquet frame from the preprocessed dataset."""
    frame_path = require_existing_path(base_dir / relative_path, f"cached frame {relative_path}")
    return pd.read_parquet(frame_path)


def assert_matching_features(left, right, left_name, right_name):
    """Validate that two feature matrices use the same columns in the same order."""
    if not left.columns.equals(right.columns):
        missing_from_right = list(left.columns.difference(right.columns))
        missing_from_left = list(right.columns.difference(left.columns))
        raise ValueError(
            f"Feature columns differ between {left_name} and {right_name}. "
            f"Missing from {right_name}: {missing_from_right}; missing from {left_name}: {missing_from_left}."
        )


def assert_index_aligned(features, targets, split_name):
    """Validate feature/target row alignment."""
    if not features.index.equals(targets.index):
        raise ValueError(f"Cached {split_name} features and targets have different indexes.")


def assert_target_columns(targets, split_name):
    """Validate target columns and order."""
    if list(targets.columns) != TARGET_COLUMNS:
        raise ValueError(
            f"Cached {split_name} target columns are {list(targets.columns)!r}; "
            f"expected {TARGET_COLUMNS!r}."
        )


def load_cached_validation_data(metadata=None):
    """Load cached train/validation matrices for fair model comparison."""
    if metadata is None:
        metadata = load_preprocessed_metadata()
    validation_dir = require_existing_path(PREPROCESSED_DATA_DIR / "validation", "validation data directory")

    X_train = load_cached_frame("validation/X_train.parquet")
    y_train = load_cached_frame("validation/y_train.parquet")
    X_val = load_cached_frame("validation/X_val.parquet")
    y_val = load_cached_frame("validation/y_val.parquet")

    assert_index_aligned(X_train, y_train, "validation train")
    assert_index_aligned(X_val, y_val, "validation holdout")
    assert_matching_features(X_train, X_val, "X_train", "X_val")
    assert_target_columns(y_train, "validation train")
    assert_target_columns(y_val, "validation holdout")

    print("Loaded cached validation data from:", validation_dir)
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    print("Feature column count:", X_train.shape[1])
    return metadata, X_train, y_train, X_val, y_val


def load_cached_final_data():
    """Load cached full-train and competition-test matrices for submission."""
    final_dir = require_existing_path(PREPROCESSED_DATA_DIR / "final", "final data directory")
    scaler_path = require_existing_path(final_dir / "gps_scaler.joblib", "final GPS scaler")
    categorical_encoder_path = require_existing_path(
        final_dir / "categorical_encoder.joblib",
        "final categorical encoder",
    )

    X_final_train = load_cached_frame("final/X_train.parquet")
    y_final_train = load_cached_frame("final/y_train.parquet")
    X_final_test = load_cached_frame("final/X_test.parquet")
    final_gps_scaler = joblib.load(scaler_path)
    final_categorical_encoder = joblib.load(categorical_encoder_path)

    assert_index_aligned(X_final_train, y_final_train, "final train")
    assert_matching_features(X_final_train, X_final_test, "X_final_train", "X_final_test")
    assert_target_columns(y_final_train, "final train")

    print("Loaded cached final data from:", final_dir)
    print("X_final_train shape:", X_final_train.shape)
    print("y_final_train shape:", y_final_train.shape)
    print("X_final_test shape:", X_final_test.shape)
    return X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder

def extract_trip_group(index_value, separator="__prefix_"):
    """Recover the original trip identifier from one cached prefix row."""
    value = str(index_value)
    if separator not in value:
        raise ValueError(
            f"Cached prefix index {value!r} does not contain {separator!r}."
        )
    return value.rsplit(separator, 1)[0]


def export_validation_predictions(
    features,
    targets,
    predictions,
    metadata,
    model_configuration,
    reviewed_validation_mean_km,
    source_notebook,
):
    """Write aligned validation predictions and a provenance sidecar."""
    row_ids = pd.Index(features.index.astype(str), name="ROW_ID")
    if not row_ids.is_unique:
        raise ValueError("Validation prediction export requires unique ROW_ID values.")
    if not features.index.equals(targets.index):
        raise ValueError("Validation features and targets are not index-aligned.")

    truth = np.asarray(targets[TARGET_COLUMNS].to_numpy(), dtype=float)
    prediction_matrix = np.asarray(predictions, dtype=float)
    expected_shape = (len(row_ids), len(TARGET_COLUMNS))
    if truth.shape != expected_shape or prediction_matrix.shape != expected_shape:
        raise ValueError(
            "Validation export shape mismatch: "
            f"truth={truth.shape}, predictions={prediction_matrix.shape}, "
            f"expected={expected_shape}."
        )
    if not np.isfinite(truth).all() or not np.isfinite(prediction_matrix).all():
        raise ValueError("Validation export contains non-finite coordinates.")

    trip_groups = [extract_trip_group(value) for value in row_ids]
    export_frame = pd.DataFrame({
        "ROW_ID": row_ids,
        "TRIP_GROUP": trip_groups,
        "END_LONG": truth[:, 0],
        "END_LAT": truth[:, 1],
        "PRED_END_LONG": prediction_matrix[:, 0],
        "PRED_END_LAT": prediction_matrix[:, 1],
    })
    if export_frame.isna().any().any():
        raise ValueError("Validation prediction export contains missing values.")

    export_metrics, _ = evaluate_long_lat_predictions(
        truth,
        prediction_matrix,
        prefix="export",
    )
    export_mean_km = float(export_metrics["export/haversine_mean_km"])
    reviewed_drift_km = export_mean_km - float(reviewed_validation_mean_km)
    if abs(reviewed_drift_km) > VALIDATION_EXPORT_DRIFT_TOLERANCE_KM:
        raise ValueError(
            f"{MODEL_NAME} validation export drifted by {reviewed_drift_km:.6f} km "
            f"from the reviewed score {reviewed_validation_mean_km:.12f} km."
        )

    export_frame.to_parquet(
        VALIDATION_PREDICTIONS_PATH,
        index=False,
        compression="snappy",
    )
    split_config = metadata.get("config", {})
    manifest = {
        "schema_version": 1,
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "source_notebook": source_notebook,
        "prediction_file": VALIDATION_PREDICTIONS_PATH.name,
        "row_count": int(len(export_frame)),
        "columns": list(export_frame.columns),
        "target_order": TARGET_COLUMNS.copy(),
        "model_configuration": model_configuration,
        "preprocessed_dataset": {
            "schema_version": metadata.get("schema_version"),
            "validation_shape": metadata.get("validation", {}).get("X_val_shape"),
            "config": {
                key: split_config.get(key)
                for key in [
                    "seed",
                    "validation_size",
                    "validation_train_seed",
                    "validation_val_seed",
                    "target_format",
                ]
            },
        },
        "validation_metrics": {
            "mean_haversine_km": export_mean_km,
            "reviewed_mean_haversine_km": float(reviewed_validation_mean_km),
            "drift_from_reviewed_km": float(reviewed_drift_km),
        },
        "kaggle": {
            "kernel_run_id": os.environ.get("KAGGLE_KERNEL_RUN_ID"),
            "kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        },
    }
    save_json_artifact(manifest, VALIDATION_PREDICTIONS_MANIFEST_PATH)
    return manifest


## Evaluation And Submission Helpers

Evaluate validation predictions and write the final Kaggle submission file.


In [ ]:
def haversine(pred, gt):
    """Return row-wise Haversine distances in kilometers for [LAT, LONG] arrays."""
    pred_lat = np.radians(pred[:, 0])
    pred_long = np.radians(pred[:, 1])
    gt_lat = np.radians(gt[:, 0])
    gt_long = np.radians(gt[:, 1])

    dlat = gt_lat - pred_lat
    dlon = gt_long - pred_long

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(pred_lat) * np.cos(gt_lat) * np.sin(dlon / 2) ** 2
    )

    d = 2 * 6371 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return d

def long_lat_to_lat_long(values):
    """Convert [LONG, LAT] arrays to [LAT, LONG] arrays."""
    values = np.asarray(values, dtype=float)
    return np.column_stack([values[:, 1], values[:, 0]])

def evaluate_long_lat_predictions(y_true_long_lat, y_pred_long_lat, prefix="val"):
    """Evaluate [LONG, LAT] predictions with Haversine distances."""
    distances = haversine(
        pred=long_lat_to_lat_long(y_pred_long_lat),
        gt=long_lat_to_lat_long(y_true_long_lat),
    )
    return {
        f"{prefix}/haversine_mean_km": float(distances.mean()),
        f"{prefix}/haversine_median_km": float(np.median(distances)),
        f"{prefix}/haversine_std_km": float(distances.std()),
        f"{prefix}/haversine_p90_km": float(np.percentile(distances, 90)),
    }, distances

def write_submission(trip_ids, destinations, output_path):
    """Write a Kaggle submission CSV from [LAT, LONG] destinations."""
    n_samples = len(trip_ids)
    assert destinations.shape == (n_samples, 2)

    submission = pd.DataFrame(
        data={
            'LATITUDE': destinations[:, 0],
            'LONGITUDE': destinations[:, 1],
        },
        columns=["LATITUDE", "LONGITUDE"],
        index=trip_ids,
    )

    submission.to_csv(output_path, index_label="TRIP_ID")


def make_json_safe(value):
    """Convert common notebook values to strict JSON-safe objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value

def save_json_artifact(payload, output_path):
    """Write an indented JSON artifact to Kaggle output storage."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(make_json_safe(payload), file, indent=2, sort_keys=True, allow_nan=False)

def save_best_params_artifact(tuning_results, validation_metrics, execution_status):
    """Persist active parameters and their validation context."""
    payload = {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "params_source": "tuning_results" if not tuning_results.empty else "BEST_MODEL_PARAMS",
        "saved_best_model_params": BEST_MODEL_PARAMS,
        "best_model_run_summary": BEST_MODEL_RUN_SUMMARY,
        "active_model_params": MODEL_PARAMS,
        "validation_metrics": validation_metrics,
        "execution": execution_status,
        "preprocessed_dataset": {
            "path": str(PREPROCESSED_DATA_DIR),
            "schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "config": PREPROCESSED_METADATA.get("config", {}),
            "validation": PREPROCESSED_METADATA.get("validation", {}),
            "final": PREPROCESSED_METADATA.get("final", {}),
        },
        "tuning": {
            "run_hyperparameter_tuning": RUN_HYPERPARAMETER_TUNING,
            "strategy": "finite_search",
            "max_trials": TUNING_MAX_TRIALS,
            "candidate_index": TUNING_CANDIDATE_INDEX,
            "candidate_label": (
                None
                if tuning_results.empty
                else tuning_results.sort_values("val/haversine_mean_km", ascending=True)
                .iloc[0]
                .get("candidate_label")
            ),
            "unique_candidate_count": TUNING_UNIQUE_CANDIDATE_COUNT,
            "search_space": TUNING_SEARCH_SPACE,
        },
    }
    if not tuning_results.empty:
        best_trial = tuning_results.sort_values("val/haversine_mean_km", ascending=True).iloc[0]
        payload["best_trial"] = best_trial.to_dict()

    save_json_artifact(payload, BEST_PARAMS_PATH)
    return payload

## Cached Validation Dataset Loading

Load shared train/validation matrices only when tuning or validation refitting is enabled.


In [ ]:
PREPROCESSED_METADATA = load_preprocessed_metadata()
needs_validation_data = RUN_HYPERPARAMETER_TUNING or RUN_VALIDATION_REFIT

if needs_validation_data:
    _, X_train, y_train, X_val, y_val = load_cached_validation_data(PREPROCESSED_METADATA)
    validation_metadata = PREPROCESSED_METADATA.get("validation", {})
    prefix_summary = pd.DataFrame(validation_metadata.get("prefix_summary", {}))
    if not prefix_summary.empty:
        display(prefix_summary)
else:
    X_train = y_train = X_val = y_val = None
    print("Skipping cached validation matrices because tuning and validation refit are disabled.")

## Define Random Forest Model

Build the estimator used by validation, tuning, and final submission.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def build_model(params=None):
    """Build the Random Forest estimator with active parameters."""
    model_params = (params or MODEL_PARAMS).copy()
    model_params["random_state"] = SEED
    model_params["n_jobs"] = -1
    return RandomForestRegressor(**model_params)


## Hyperparameter Tuning

Optional tuning is disabled by default; saved best parameters are used unless explicitly enabled.


### Tuning Logging Helpers

Validate W&B once before optional tuning. Missing or invalid credentials disable logging, and per-trial SDK failures never interrupt model evaluation.

In [ ]:
WANDB_ENABLED = False
_WANDB_MODULE = None
_WANDB_MODE = "disabled"

def get_wandb_key():
    """Return the W&B API key from Kaggle secrets or the environment."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    except Exception:
        return os.environ.get("WANDB_API_KEY")

def _wandb_warning(stage, exception):
    """Report a sanitized W&B failure without exposing credentials."""
    print(
        f"W&B {stage} failed ({type(exception).__name__}). "
        "Model tuning will continue."
    )

def initialize_wandb():
    """Validate W&B credentials once before optional tuning starts."""
    wandb_key = get_wandb_key()
    if not wandb_key:
        print("No W&B key found. Tuning will run without W&B logging.")
        return None, "disabled", False

    try:
        import wandb
        login_succeeded = wandb.login(key=wandb_key)
        if login_succeeded is False:
            raise RuntimeError("W&B login was rejected.")
    except Exception as exception:
        _wandb_warning("setup", exception)
        return None, "disabled", False

    return wandb, "online", True

def sanitize_for_wandb(value):
    """Convert values to W&B-safe scalar or container types."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, dict):
        return {key: sanitize_for_wandb(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_for_wandb(item) for item in value]
    return value

def log_tuning_trial_to_wandb(result):
    """Log one tuning trial without allowing W&B to interrupt tuning."""
    if not WANDB_ENABLED:
        return

    run = None
    try:
        trial_index = int(result["trial_index"])
        params = sanitize_for_wandb(result["params"])
        tuner_type = result["tuner_type"]
        candidate_label = result["candidate_label"]
        preprocessed_config = PREPROCESSED_METADATA.get("config", {})
        config = {
            "model_slug": MODEL_SLUG,
            "trial_index": trial_index,
            "candidate_label": candidate_label,
            "trial_params": params,
            "seed": SEED,
            "tuner_type": tuner_type,
            "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "preprocessed_n_rows": preprocessed_config.get("n_rows"),
            "preprocessed_validation_size": preprocessed_config.get("validation_size"),
            "preprocessed_prefix_sample_factor": preprocessed_config.get("prefix_sample_factor"),
        }

        low_importance_metric_keys = {
            "train/haversine_median_km",
            "train/haversine_p90_km",
            "train/haversine_std_km",
        }
        metrics = {
            key: sanitize_for_wandb(value)
            for key, value in result.items()
            if key not in {"params", "params_repr", "tuner_type", "candidate_label"}
            and key not in low_importance_metric_keys
        }

        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=f"{MODEL_SLUG}-{candidate_label}-{tuner_type}-trial-{trial_index:03d}",
            mode=_WANDB_MODE,
            config=config,
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no run.")
        _WANDB_MODULE.log(metrics)
    except Exception as exception:
        _wandb_warning("trial logging", exception)
    finally:
        if run is not None:
            try:
                run.finish()
            except Exception as exception:
                _wandb_warning("run finalization", exception)

if RUN_HYPERPARAMETER_TUNING:
    _WANDB_MODULE, _WANDB_MODE, WANDB_ENABLED = initialize_wandb()

### Tuning Candidate Evaluation

Fit the explicitly selected candidate and collect validation-only diagnostics.

In [ ]:
def evaluate_tuning_candidate(
    candidate_params,
    candidate_index,
    candidate_label,
):
    """Fit one finite candidate and return validation-only metrics."""
    candidate_model = build_model(params=candidate_params)

    start_time = time.time()
    candidate_model.fit(X_train, y_train)
    fit_seconds = time.time() - start_time

    val_predictions = candidate_model.predict(X_val)
    val_metrics, _ = evaluate_long_lat_predictions(
        y_val.values,
        val_predictions,
        prefix="val",
    )

    return {
        "trial_index": candidate_index,
        "candidate_label": candidate_label,
        "params": candidate_params.copy(),
        "params_repr": repr(candidate_params),
        "tuner_type": "finite_search",
        **val_metrics,
        "fit_seconds": float(fit_seconds),
    }

### One-Candidate Finite Tuning

Select one deterministic configuration by its one-based index for this Kaggle run.

In [ ]:
def format_candidate_label(params):
    """Return a stable label for one Random Forest candidate."""
    depth_label = "none" if params["max_depth"] is None else str(params["max_depth"])
    max_features_label = str(params["max_features"]).replace(".", "")
    return (
        f"rf_n{params['n_estimators']}_depth{depth_label}_"
        f"mf{max_features_label}"
    )


def get_finite_candidate_configs():
    """Return the deterministic 12-configuration finite region."""
    candidates = list(ParameterGrid(TUNING_SEARCH_SPACE))
    if len(candidates) != TUNING_UNIQUE_CANDIDATE_COUNT:
        raise RuntimeError("Finite candidate count changed unexpectedly.")
    return candidates


def run_finite_search_tuning():
    """Evaluate exactly one indexed finite candidate for this Kaggle run."""
    global MODEL_PARAMS

    candidates = get_finite_candidate_configs()
    candidate_params = candidates[TUNING_CANDIDATE_INDEX - 1]
    candidate_label = format_candidate_label(candidate_params)
    print(
        f"Evaluating candidate {TUNING_CANDIDATE_INDEX}/"
        f"{TUNING_UNIQUE_CANDIDATE_COUNT}: {candidate_params}"
    )

    tuning_start_time = time.time()
    result = evaluate_tuning_candidate(
        candidate_params=candidate_params,
        candidate_index=TUNING_CANDIDATE_INDEX,
        candidate_label=candidate_label,
    )
    result["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
    log_tuning_trial_to_wandb(result)

    MODEL_PARAMS = result["params"].copy()
    tuning_results = pd.DataFrame([
        {key: value for key, value in result.items() if key != "params"}
    ])

    print(f"  val={result['val/haversine_mean_km']:.4f} km")
    print(
        "Finite candidate validation Haversine: "
        f"{result['val/haversine_mean_km']:.6f} km"
    )
    import pprint
    print("Paste-ready best config:")
    print("BEST_MODEL_PARAMS = " + pprint.pformat(MODEL_PARAMS, sort_dicts=False))
    display(tuning_results)
    return tuning_results

### Run Hyperparameter Tuning

Use saved best parameters unless tuning is explicitly enabled.


In [ ]:
def run_hyperparameter_tuning():
    """Evaluate one finite candidate or keep the saved parameters."""
    global MODEL_PARAMS

    if not RUN_HYPERPARAMETER_TUNING:
        MODEL_PARAMS = BEST_MODEL_PARAMS.copy()
        print("Hyperparameter tuning disabled. Using BEST_MODEL_PARAMS:")
        print(MODEL_PARAMS)
        return pd.DataFrame()

    return run_finite_search_tuning()


tuning_results = run_hyperparameter_tuning()
print("Active MODEL_PARAMS:", MODEL_PARAMS)

## Train And Validate

Refit the selected model on the validation-training split when enabled; otherwise reuse the current finite-candidate metrics or the reviewed model summary.

In [ ]:
validation_export_manifest = None

if RUN_VALIDATION_REFIT:
    model = build_model()
    print(model)

    start_time = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start_time

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)

    train_metrics, _ = evaluate_long_lat_predictions(
        y_train.values,
        train_predictions,
        prefix="train",
    )
    val_metrics, val_distances = evaluate_long_lat_predictions(
        y_val.values,
        val_predictions,
        prefix="val",
    )

    validation_metrics = {
        **train_metrics,
        **val_metrics,
        "generalization/haversine_gap_km": float(
            val_metrics["val/haversine_mean_km"]
            - train_metrics["train/haversine_mean_km"]
        ),
        "fit_seconds": float(fit_seconds),
        "validation_metrics_source": "current_validation_refit",
    }
    validation_metrics_source = "current_validation_refit"
    validation_export_manifest = export_validation_predictions(
        features=X_val,
        targets=y_val,
        predictions=val_predictions,
        metadata=PREPROCESSED_METADATA,
        model_configuration=MODEL_PARAMS.copy(),
        reviewed_validation_mean_km=BEST_MODEL_RUN_SUMMARY["validation_haversine_mean_km"],
        source_notebook="competition_random_forest.ipynb",
    )
elif not tuning_results.empty:
    best_tuning_trial = tuning_results.iloc[0]
    validation_metrics = {
        "val/haversine_mean_km": float(
            best_tuning_trial["val/haversine_mean_km"]
        ),
        "val/haversine_median_km": float(
            best_tuning_trial["val/haversine_median_km"]
        ),
        "val/haversine_std_km": float(
            best_tuning_trial["val/haversine_std_km"]
        ),
        "val/haversine_p90_km": float(
            best_tuning_trial["val/haversine_p90_km"]
        ),
        "fit_seconds": float(best_tuning_trial["fit_seconds"]),
        "validation_metrics_source": "current_finite_search_candidate",
    }
    validation_metrics_source = "current_finite_search_candidate"
    val_distances = None
    print("Using validation metrics from the current finite candidate.")
else:
    train_metrics = {
        "train/haversine_mean_km": BEST_MODEL_RUN_SUMMARY["train_haversine_mean_km"],
        "train/haversine_median_km": BEST_MODEL_RUN_SUMMARY["train_haversine_median_km"],
        "train/haversine_std_km": BEST_MODEL_RUN_SUMMARY["train_haversine_std_km"],
        "train/haversine_p90_km": BEST_MODEL_RUN_SUMMARY["train_haversine_p90_km"],
    }
    val_metrics = {
        "val/haversine_mean_km": BEST_MODEL_RUN_SUMMARY["validation_haversine_mean_km"],
        "val/haversine_median_km": BEST_MODEL_RUN_SUMMARY["validation_haversine_median_km"],
        "val/haversine_std_km": BEST_MODEL_RUN_SUMMARY["validation_haversine_std_km"],
        "val/haversine_p90_km": BEST_MODEL_RUN_SUMMARY["validation_haversine_p90_km"],
    }
    validation_metrics = {
        **train_metrics,
        **val_metrics,
        "generalization/haversine_gap_km": BEST_MODEL_RUN_SUMMARY["generalization_gap_km"],
        "fit_seconds": BEST_MODEL_RUN_SUMMARY["diagnostic_refit_fit_seconds"],
        "validation_metrics_source": "reviewed_best_model_run_summary",
    }
    validation_metrics_source = "reviewed_best_model_run_summary"
    val_distances = None
    print(
        "Skipping validation refit. Using reviewed validation metrics "
        "from BEST_MODEL_RUN_SUMMARY."
    )

print(pd.Series(validation_metrics))

execution_status = {
    "validation_refit_requested": RUN_VALIDATION_REFIT,
    "validation_refit_completed": bool(RUN_VALIDATION_REFIT),
    "validation_metrics_source": validation_metrics_source,
    "final_fit_requested": RUN_FINAL_FIT,
    "final_fit_completed": False,
    "model_bundle_written": False,
    "model_serialization_requested": bool(SERIALIZE_RANDOM_FOREST_MODEL),
    "model_serialized": False,
    "model_serialization_error": None,
    "submission_written": False,
}

if validation_export_manifest is not None:
    execution_status.update({
        "validation_predictions_written": True,
        "validation_predictions_path": str(VALIDATION_PREDICTIONS_PATH),
        "validation_predictions_manifest_path": str(VALIDATION_PREDICTIONS_MANIFEST_PATH),
        "validation_prediction_rows": validation_export_manifest["row_count"],
    })
else:
    execution_status["validation_predictions_written"] = False

if enabled_expensive_stages:
    save_best_params_artifact(
        tuning_results,
        validation_metrics,
        execution_status,
    )
    print("Best params JSON written to:", BEST_PARAMS_PATH.resolve())
else:
    print("Inspection mode: no artifacts were written.")

## Train vs Validation Plot

When validation refitting is enabled, compare train and validation mean errors and show the real validation-distance histogram.


In [ ]:
if RUN_VALIDATION_REFIT:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    mean_values = [
        train_metrics["train/haversine_mean_km"],
        val_metrics["val/haversine_mean_km"],
    ]
    bars = axes[0].bar(["Train", "Validation"], mean_values, color=["#4C78A8", "#F58518"])
    axes[0].set_ylabel("Mean Haversine distance (km)")
    axes[0].set_title(f"{MODEL_NAME} train vs validation error")
    axes[0].bar_label(bars, fmt="%.3f km")

    axes[1].hist(val_distances, bins=40, color="#F58518")
    axes[1].axvline(
        val_distances.mean(),
        color="red",
        linestyle="--",
        label=f"Mean: {val_distances.mean():.3f} km",
    )
    axes[1].set_xlabel("Haversine distance (km)")
    axes[1].set_ylabel("Validation trips")
    axes[1].set_title(f"{MODEL_NAME} validation error distribution")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Skipping validation plots because RUN_VALIDATION_REFIT=False.")

for name in [
    "model",
    "train_predictions",
    "val_predictions",
    "val_distances",
    "validation_export_manifest",
    "X_train",
    "y_train",
    "X_val",
    "y_val",
]:
    globals().pop(name, None)
gc.collect()

## Final Fit And Kaggle Submission

When RUN_FINAL_FIT=True, load cached full-train and test matrices, fit the final model, write a lightweight reconstruction bundle by default, and export the submission. Set SERIALIZE_RANDOM_FOREST_MODEL=True only when the fitted estimator must be included.


In [ ]:
if RUN_FINAL_FIT:
    X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder = load_cached_final_data()

    final_model = build_model()
    final_model.fit(X_final_train, y_final_train)

    execution_status["final_fit_completed"] = True
    save_best_params_artifact(
        tuning_results,
        validation_metrics,
        execution_status,
    )

    selected_best_run_summary = BEST_MODEL_RUN_SUMMARY.copy()
    selected_best_trial_index = int(selected_best_run_summary["trial_index"])
    equivalent_best_trial_indices = selected_best_run_summary.get(
        "equivalent_best_trial_indices",
        [selected_best_trial_index],
    ).copy()

    def build_model_bundle(bundle_model, model_serialized):
        """Build the lightweight or estimator-inclusive final bundle."""
        return {
            "model": bundle_model,
            "model_serialized": model_serialized,
            "model_omitted_reason": (
                None
                if model_serialized
                else "The fitted Random Forest estimator is omitted from this lightweight safety bundle; rebuild it from cached preprocessing data and the saved model parameters if optional serialization does not complete."
            ),
            "gps_scaler": final_gps_scaler,
            "categorical_encoder": final_categorical_encoder,
            "feature_columns": list(X_final_train.columns),
            "model_params": {
                **MODEL_PARAMS,
                "random_state": SEED,
                "n_jobs": -1,
            },
            "validation_metrics": validation_metrics.copy(),
            "selected_best_run_summary": selected_best_run_summary,
            "selected_best_trial_index": selected_best_trial_index,
            "equivalent_best_trial_indices": equivalent_best_trial_indices,
            "final_refit": True,
            "final_refit_note": "Model is refit on final training data with the selected best parameters; the fitted Random Forest estimator is serialized only when SERIALIZE_RANDOM_FOREST_MODEL is True.",
            "prediction_format": "[LONG, LAT]",
            "preprocessed_data_dir": str(PREPROCESSED_DATA_DIR),
            "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "preprocessed_metadata": PREPROCESSED_METADATA,
        }

    lightweight_bundle = build_model_bundle(
        bundle_model=None,
        model_serialized=False,
    )
    joblib.dump(lightweight_bundle, MODEL_BUNDLE_PATH)
    execution_status["model_bundle_written"] = True
    save_best_params_artifact(
        tuning_results,
        validation_metrics,
        execution_status,
    )
    print("Lightweight model bundle written to:", MODEL_BUNDLE_PATH.resolve())

    test_predictions = final_model.predict(X_final_test)

    submission_path = SUBMISSION_DIR / SUBMISSION_FILE
    test_destinations = long_lat_to_lat_long(test_predictions)
    write_submission(X_final_test.index, test_destinations, submission_path)
    submission = pd.read_csv(submission_path)

    execution_status["submission_written"] = True
    save_best_params_artifact(
        tuning_results,
        validation_metrics,
        execution_status,
    )
    print("Submission written to:", submission_path.resolve())
    display(submission.head())

    if SERIALIZE_RANDOM_FOREST_MODEL:
        serialized_bundle_path = MODEL_BUNDLE_PATH.with_suffix(
            MODEL_BUNDLE_PATH.suffix + ".tmp"
        )
        try:
            serialized_bundle = build_model_bundle(
                bundle_model=final_model,
                model_serialized=True,
            )
            joblib.dump(serialized_bundle, serialized_bundle_path)
            os.replace(serialized_bundle_path, MODEL_BUNDLE_PATH)
            execution_status["model_serialized"] = True
            save_best_params_artifact(
                tuning_results,
                validation_metrics,
                execution_status,
            )
            print(
                "Serialized model bundle written to:",
                MODEL_BUNDLE_PATH.resolve(),
            )
        except Exception as exc:
            execution_status["model_serialization_error"] = (
                f"{type(exc).__name__}: {exc}"
            )
            save_best_params_artifact(
                tuning_results,
                validation_metrics,
                execution_status,
            )
            raise
        finally:
            serialized_bundle_path.unlink(missing_ok=True)
else:
    print("Skipping final fit because RUN_FINAL_FIT=False.")